# NLP Classification of Technical Newsgroup Posts - Modeling
notebook 2

Table of Content

- [Imports and Setup](#imports-and-setup)
- [Split strategy and leakage control](#split-strategy-and-leakage-control)
- 



## <a id="imports-and-setup"></a>Imports and Setup


This notebook continues from the EDA and preprocessing notebook.

It loads the processed 20 Newsgroups technical subset, including all text regimes, artifact diagnostics, and evaluation slices created during EDA.

The main goals of this notebook are:

1. Define a leakage-aware train/validation/test split strategy.
2. Train classical NLP baselines across text regimes.
3. Compare performance using macro F1, per-class metrics, confusion matrices, and slice analysis.
4. Inspect model features to check whether predictions rely on topic vocabulary or shortcut artifacts.
5. Prepare a comparable transformer evaluation.

# PLAN


## Split audit
## Choose/save split
## Baseline MVP: TF-IDF + LR on text_subject_body_clean
## Text regime comparison
## TF-IDF + NB/LR ablation
## Feature inspection
## Slice evaluation
## Error analysis
## Transformer
## Transformer error/explainability
## Final recommendation

In [2]:
# standard library 
import itertools
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path

# core data 
import numpy as np
import pandas as pd

# visualisation
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from sklearn.metrics.pairwise import cosine_similarity
import random
from scipy.sparse import vstack

import platform

In [3]:
# global configs

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

In [4]:
PROJECT_ROOT = Path(".")

DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_CONFIG_DIR = PROJECT_ROOT / "outputs" / "config"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

MODELING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "modeling"
MODELING_TABLES_DIR = MODELING_OUTPUT_DIR / "tables"
MODELING_CONFIG_DIR = MODELING_OUTPUT_DIR / "config"
MODELING_FIGURES_DIR = MODELING_OUTPUT_DIR / "figures"
MODELING_PREDICTIONS_DIR = MODELING_OUTPUT_DIR / "predictions"

for directory in [
    MODELING_OUTPUT_DIR,
    MODELING_TABLES_DIR,
    MODELING_CONFIG_DIR,
    MODELING_FIGURES_DIR,
    MODELING_PREDICTIONS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

In [5]:
# load EAD manifest and config

manifest_path = OUTPUT_CONFIG_DIR / "eda_output_manifest.json"
config_path = OUTPUT_CONFIG_DIR / "eda_preprocessing_config.json"

with open(manifest_path, "r", encoding="utf-8") as f:
    eda_manifest = json.load(f)

with open(config_path, "r", encoding="utf-8") as f:
    preprocessing_config = json.load(f)

print("Loaded EDA manifest:")
print(json.dumps(eda_manifest, indent=2, ensure_ascii=False)[:1500])

print("\nLoaded preprocessing config keys:")
print(preprocessing_config.keys())

Loaded EDA manifest:
{
  "created_at": "2026-05-27T22:54:32",
  "processed_dataset": "data/processed/20ng_tech_preprocessed.parquet",
  "modeling_input_dataset": "data/processed/20ng_tech_modeling_input.parquet",
  "config": "outputs/config/eda_preprocessing_config.json",
  "environment": "outputs/config/eda_environment_info.json",
  "saved_tables": {
    "length_impact": "outputs/tables/preprocessing_length_impact.csv",
    "retention_summary": "outputs/tables/preprocessing_word_retention_summary.csv",
    "class_length_impact": "outputs/tables/preprocessing_class_length_impact.csv",
    "artifact_overall": "outputs/tables/artifact_overall.csv",
    "artifact_by_class": "outputs/tables/artifact_density_by_class.csv",
    "artifact_intensity_selected": "outputs/tables/artifact_intensity_selected.csv",
    "artifact_residuals": "outputs/tables/artifact_residuals_after_preprocessing.csv",
    "length_slice_by_regime": "outputs/tables/length_slice_by_regime.csv",
    "length_slice_pivot":

In [6]:
# load processed modeling dataset

modeling_input_path = Path(eda_manifest["modeling_input_dataset"])

if modeling_input_path.suffix == ".parquet":
    df = pd.read_parquet(modeling_input_path)
elif modeling_input_path.suffix == ".pkl":
    df = pd.read_pickle(modeling_input_path)
else:
    raise ValueError(f"Unsupported modelling input format: {modeling_input_path}")

df.shape

(6937, 37)

In [7]:
df.head()

,doc_id,original_index,filename,message_id,label,label_id,text,text_raw,text_metadata_rich_no_leakage,text_subject_body_clean,text_body_only_clean,text_subject_body_no_quotes,quote_density_slice,artifact_density_slice,text_raw_length_slice,text_metadata_rich_no_leakage_length_slice,text_subject_body_clean_length_slice,text_body_only_clean_length_slice,text_subject_body_no_quotes_length_slice,body_quote_line_ratio,structural_line_ratio,has_uuencoded_payload,uuencoded_line_count,uuencoded_line_ratio,has_faq_digest_signal,is_multipart_series,has_archive_name_header,has_last_modified_header,has_approved_header,has_followup_to_poster,subject_mentions_faq,subject_mentions_digest_periodic,subject_has_part_pattern,language_or_encoding_review_flag,non_ascii_char_ratio,alphabetic_char_ratio,english_stopword_token_ratio
0,doc_00000,0,15738,<1993Apr21.184113.3505@nmt.edu>,sci.crypt,5,"Xref: cantaloupe.srv.cs.cmu.edu sci.crypt:15738 alt.security:10080 comp.org.eff.talk:17114 comp.security.misc:3535 comp.org.acm:1694 comp.org.ieee:1623\nNewsgroups: sci.crypt,alt.security,comp.org...","Xref: cantaloupe.srv.cs.cmu.edu sci.crypt:15738 alt.security:10080 comp.org.eff.talk:17114 comp.security.misc:3535 comp.org.acm:1694 comp.org.ieee:1623\nNewsgroups: sci.crypt,alt.security,comp.org...",Xref: cantaloupe.srv.cs.cmu.edu sci.crypt:15738 alt.security:10080 comp.org.eff.talk:17114 comp.security.misc:3535 comp.org.acm:1694 comp.org.ieee:1623\nPath: cantaloupe.srv.cs.cmu.edu!magnesium.c...,public awareness (wasRe: text of White House announcement and Q&As on clipper chip encryption)\n\nIn article < <EMAIL> > <EMAIL> (Pat Myrto) writes:\n>I think this is no accident. It comes from th...,"In article < <EMAIL> > <EMAIL> (Pat Myrto) writes:\n>I think this is no accident. It comes from the same philosophy that\n>the government rules/controls the people, not the people controlling\n>th...",public awareness (wasRe: text of White House announcement and Q&As on clipper chip encryption)\n\nIn article < <EMAIL> > <EMAIL> (Pat Myrto) writes:\n\nThe average amerikan today seems to think th...,quote_dominated_50%plus,high_structural_artifacts,very_long_512plus,very_long_512plus,long_256_511,long_256_511,medium_80_255,0.620000,0.492063,False,0,0.0,False,False,False,False,False,False,False,False,False,False,0.0,0.753913,0.452101
1,doc_00001,1,60553,<1993Apr19.185511.2072@icd.teradyne.com>,comp.sys.ibm.pc.hardware,2,Newsgroups: comp.sys.ibm.pc.hardware\nPath: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!howland.reston.ans.net!bogus.sura.net!...,Newsgroups: comp.sys.ibm.pc.hardware\nPath: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!howland.reston.ans.net!bogus.sura.net!...,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!howland.reston.ans.net!bogus.sura.net!news-feed-1.peachnet.edu!umn.edu!csus....,"Re: Recommendations for a Local BUS (Cache\n\nIn article <EMAIL> , <EMAIL> (Penio Penev) writes:\n>On 15 Apr 1993 20:14:20 GMT Divya Sundaram ( <EMAIL> ) wrote:\n>\n>| I would like to hear the net...","In article <EMAIL> , <EMAIL> (Penio Penev) writes:\n>On 15 Apr 1993 20:14:20 GMT Divya Sundaram ( <EMAIL> ) wrote:\n>\n>| I would like to hear the net.wisdom and net.opinions on IDE Controllers.\n...","Re: Recommendations for a Local BUS (Cache\n\nIn article <EMAIL> , <EMAIL> (Penio Penev) writes:\n\nI also have a DX2/66 and a Maxtor 212. I have a local bus IDE controller (generic) and I get\n98...",quote_dominated_50%plus,high_structural_artifacts,long_256_511,long_256_511,long_256_511,long_256_511,medium_80_255,0.500000,0.372549,False,0,0.0,False,False,False,False,False,False,False,False,False,False,0.0,0.720565,0.344920
2,doc_00002,2,58989,<1pstlnINN4r5@srvr1.engin.umich.edu>,comp.sys.ibm.pc.hardware,2,Path: cantaloupe.srv.cs.cmu.edu!das-news.harvard.edu!ogicse!uwm.e

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6937 entries, 0 to 6936
Data columns (total 37 columns):
 #   Column                                      Non-Null Count  Dtype   
---  ------                                      --------------  -----   
 0   doc_id                                      6937 non-null   object  
 1   original_index                              6937 non-null   int64   
 2   filename                                    6937 non-null   object  
 3   message_id                                  6937 non-null   object  
 4   label                                       6937 non-null   object  
 5   label_id                                    6937 non-null   int64   
 6   text                                        6937 non-null   object  
 7   text_raw                                    6937 non-null   object  
 8   text_metadata_rich_no_leakage               6937 non-null   object  
 9   text_subject_body_clean                     6937 non-null   object  
 10  

In [10]:
# define text regimes and labels

# All text representations available in the dataset.
# text_raw is retained for audit and thread metadata extraction only.
TEXT_REGIMES = [
    "text_raw",
    "text_metadata_rich_no_leakage",
    "text_subject_body_clean",
    "text_body_only_clean",
    "text_subject_body_no_quotes",
]

# Regimes used for fair modelling comparisons.
# text_raw is excluded because it contains direct Newsgroups label leakage.
MODELING_TEXT_REGIMES = [
    "text_metadata_rich_no_leakage",
    "text_subject_body_clean",
    "text_body_only_clean",
    "text_subject_body_no_quotes",
]

MAIN_TEXT_REGIME = "text_subject_body_clean"
METADATA_RICH_DIAGNOSTIC_REGIME = "text_metadata_rich_no_leakage"
RAW_AUDIT_REGIME = "text_raw"

TARGET_COL = "label"
TARGET_ID_COL = "label_id"
ID_COL = "doc_id"
MESSAGE_ID_COL = "message_id"

LABELS = sorted(df[TARGET_COL].unique())

LABELS

['comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'sci.crypt',
 'sci.electronics']

In [13]:
# sanity
expected_columns = [
    ID_COL,
    MESSAGE_ID_COL,
    TARGET_COL,
    TARGET_ID_COL,
    *TEXT_REGIMES,
]

missing_columns = [col for col in expected_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print("All expected modelling columns are present.")

All expected modelling columns are present.


In [11]:
# sanity check
text_regime_summary = []

for col in TEXT_REGIMES:
    text_regime_summary.append({
        "text_regime": col,
        "missing_texts": df[col].isna().sum(),
        "empty_texts": df[col].fillna("").str.strip().eq("").sum(),
        "mean_words": df[col].fillna("").str.split().str.len().mean(),
        "median_words": df[col].fillna("").str.split().str.len().median(),
    })

text_regime_summary = pd.DataFrame(text_regime_summary)

display(
    text_regime_summary.style.format({
        "mean_words": "{:.1f}",
        "median_words": "{:.1f}",
    })
)

,text_regime,missing_texts,empty_texts,mean_words,median_words
0,text_raw,0,0,259.2,172.0
1,text_metadata_rich_no_leakage,0,0,258.9,174.0
2,text_subject_body_clean,0,0,212.9,128.0
3,text_body_only_clean,0,33,207.2,122.0
4,text_subject_body_no_quotes,0,0,176.3,94.0


In [12]:
# optional eda tables for refernce 

def load_table_if_exists(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    print(f"File not found: {path}")
    return None

eda_tables = {}

for table_name, table_path in eda_manifest.get("saved_tables", {}).items():
    path = Path(table_path)
    if path.exists():
        eda_tables[table_name] = pd.read_csv(path)
    else:
        print(f"Missing table: {table_name} -> {path}")

print(f"Loaded {len(eda_tables)} EDA tables.")
print(sorted(eda_tables.keys()))

Loaded 21 EDA tables.
['artifact_by_class', 'artifact_intensity_selected', 'artifact_overall', 'artifact_residuals', 'artifact_slice_summary', 'centroid_similarity_df', 'class_length_impact', 'length_impact', 'length_slice_by_regime', 'length_slice_pivot', 'pair_similarity_df', 'quote_by_class', 'quote_slice_summary', 'retention_summary', 'shared_terms_pairs', 'top_terms_overall_text_body_only_clean', 'top_terms_overall_text_metadata_rich_no_leakage', 'top_terms_overall_text_raw', 'top_terms_overall_text_subject_body_clean', 'top_terms_overall_text_subject_body_no_quotes', 'vocab_diagnostics']


In [14]:
# save modeling notebook config

modeling_config = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "02_modeling_evaluation.ipynb",
    "random_state": RANDOM_STATE,
    "dataset_path": str(modeling_input_path),

    "target_col": TARGET_COL,
    "target_id_col": TARGET_ID_COL,
    "id_col": ID_COL,
    "message_id_col": MESSAGE_ID_COL,

    "text_regimes_available": TEXT_REGIMES,
    "modeling_text_regimes": MODELING_TEXT_REGIMES,
    "excluded_from_fair_modelling": [RAW_AUDIT_REGIME],

    "main_text_regime": MAIN_TEXT_REGIME,
    "metadata_rich_diagnostic_regime": METADATA_RICH_DIAGNOSTIC_REGIME,
    "raw_audit_regime": RAW_AUDIT_REGIME,

    "labels": LABELS,

    "raw_text_policy": (
        "text_raw is retained for audit and thread metadata extraction only. "
        "It is excluded from fair modelling comparisons because the raw archive "
        "contains direct Newsgroups label leakage."
    ),

    "planned_split_strategy": [
        "Use text_raw only to parse thread metadata such as Message-ID, References, and In-Reply-To.",
        "Attempt thread/quote-aware grouping using Message-ID, References, In-Reply-To, and quote fingerprints.",
        "If feasible, use group-aware train/validation/test split.",
        "If not feasible, use stratified split and report quoted-thread overlap as a validity threat.",
    ],

    "primary_metric": "macro_f1",
    "secondary_metrics": [
        "accuracy",
        "per_class_precision",
        "per_class_recall",
        "per_class_f1",
    ],

    "evaluation_slices": [
        "length_slice",
        "quote_density_slice",
        "artifact_density_slice",
        "has_uuencoded_payload",
        "has_faq_digest_signal",
        "language_or_encoding_review_flag",
    ],
}

modeling_config_path = MODELING_CONFIG_DIR / "modeling_config_initial.json"

with open(modeling_config_path, "w", encoding="utf-8") as f:
    json.dump(modeling_config, f, indent=2, ensure_ascii=False)

print(f"Saved modelling config to: {modeling_config_path}")

Saved modelling config to: outputs/modeling/config/modeling_config_initial.json


In [15]:
# env shapshot
environment_info = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": sklearn.__version__,
}

environment_path = MODELING_CONFIG_DIR / "modeling_environment_info.json"

with open(environment_path, "w", encoding="utf-8") as f:
    json.dump(environment_info, f, indent=2, ensure_ascii=False)

environment_info

{'created_at': '2026-05-27T23:00:30',
 'python_version': '3.13.5',
 'platform': 'macOS-26.3.1-arm64-arm-64bit-Mach-O',
 'pandas_version': '2.3.1',
 'numpy_version': '2.3.1',
 'sklearn_version': '1.7.1'}

## <a id="split-strategy-and-leakage-control"></a>Split strategy and leakage control


A standard random stratified split is not automatically sufficient for this dataset.

The main risk is not class imbalance, but possible content overlap through thread structure and quoted replies. During EDA, many posts were found to contain quoted previous messages. This means that two different documents may share parts of the same discussion thread. If related replies are split across train, validation, and test sets, a model may partially benefit from repeated quoted fragments rather than generalizing to genuinely unseen discussions.

This is a softer form of leakage than direct label leakage. The model is not necessarily seeing the label itself, but it may see repeated context from the same conversation across different splits. This could inflate validation or test performance, especially for lexical models such as TF-IDF.

To reduce this risk, I first attempt a graph-based grouping strategy. Each document is treated as a node. Edges between documents are created using two signals:

1. **Thread metadata**: `Message-ID`, `References`, and `In-Reply-To` headers, when available.
2. **Quote fingerprints**: normalized hashes of meaningful `>`-style quoted lines shared between documents.

Documents connected by either thread metadata or shared quote fingerprints are assigned to the same connected component. These components are then used as groups for a group-aware train/validation/test split, so that related posts stay in the same partition.

This approach is exploratory and will be audited before use. If the resulting groups are too sparse, too large, or too class-imbalanced for a reliable split, the fallback will be a standard stratified split with the quote/thread overlap risk reported explicitly as a validity limitation. In either case, the modelling stage will also compare preprocessing regimes, including a quote-reduced representation, to test how much performance depends on quoted context.

### Parse thread metadata  

The first grouping signal comes from top-level Usenet thread headers.

For this step, I parse only the initial header block of each raw document. Quoted or embedded headers inside the message body are intentionally ignored, because they may belong to previous messages rather than to the current document.

The goal is to extract:

- `Message-ID`: the current document identifier;
- `References`: previous messages in the thread;
- `In-Reply-To`: direct reply target, when available.

This metadata is not used as a model feature. It is used only to construct possible leakage-control groups for splitting.

This parsing step uses `text_raw` only for metadata extraction, not for modelling. The raw archive preserves `Message-ID`, `References`, and `In-Reply-To`, which are needed to construct thread-aware groups.

The saved `message_id` column from EDA is validated against the freshly parsed top-level `Message-ID` field before graph construction. This prevents silent mismatch between the deduplicated message identity and the metadata used for grouping.

In [16]:
# header parcer

def extract_top_header_block(text):
    """
    Extract the initial top-level header block from a raw Usenet post.

    The header block is assumed to end at the first blank line.
    """
    if pd.isna(text):
        return []

    lines = str(text).splitlines()
    header_lines = []

    for line in lines:
        if line.strip() == "":
            break
        header_lines.append(line)

    return header_lines


def parse_top_headers(text):
    """
    Parse top-level RFC-style headers from the initial header block.

    Handles folded continuation lines that start with whitespace.
    Returns a dictionary with lowercase header names.
    """
    header_lines = extract_top_header_block(text)

    headers = {}
    current_key = None

    for line in header_lines:
        header_match = re.match(r"^([A-Za-z][A-Za-z0-9\-]*)\s*:\s*(.*)$", line)

        if header_match:
            key = header_match.group(1).lower()
            value = header_match.group(2).strip()

            if key not in headers:
                headers[key] = value
            else:
                headers[key] = headers[key] + " " + value

            current_key = key

        elif current_key is not None and line.startswith((" ", "\t")):
            # folded header continuation
            headers[current_key] = headers[current_key] + " " + line.strip()

    return headers

In [17]:
# message id extraction helpers

MESSAGE_ID_PATTERN = re.compile(r"<[^<>]+>")


def normalize_message_id(message_id):
    """
    Normalize a Message-ID-like string.
    """
    if message_id is None or pd.isna(message_id):
        return None

    message_id = str(message_id).strip().lower()
    message_id = re.sub(r"\s+", "", message_id)

    return message_id if message_id else None


def extract_message_ids(value):
    """
    Extract all <...> message identifiers from a header value.
    """
    if value is None or pd.isna(value):
        return []

    ids = MESSAGE_ID_PATTERN.findall(str(value))
    ids = [normalize_message_id(mid) for mid in ids]
    ids = [mid for mid in ids if mid is not None]

    # preserve order while removing duplicates
    return list(dict.fromkeys(ids))


def first_or_none(items):
    return items[0] if len(items) > 0 else None

In [19]:
# Parse thread fields into dataframe columns.
# text_raw is audit-only for modelling, but it is the correct source for thread metadata extraction.

parsed_headers = df["text_raw"].apply(parse_top_headers)

df["parsed_message_id"] = parsed_headers.apply(
    lambda h: first_or_none(extract_message_ids(h.get("message-id")))
)

df["references_ids"] = parsed_headers.apply(
    lambda h: extract_message_ids(h.get("references"))
)

df["in_reply_to_ids"] = parsed_headers.apply(
    lambda h: extract_message_ids(h.get("in-reply-to"))
)

df["thread_reference_ids"] = df.apply(
    lambda row: list(dict.fromkeys(row["references_ids"] + row["in_reply_to_ids"])),
    axis=1,
)

In [20]:
# Validate parsed Message-ID against the saved Message-ID column from EDA.

if "message_id" in df.columns:
    mismatch_mask = (
        df["message_id"].notna()
        & df["parsed_message_id"].notna()
        & (df["message_id"] != df["parsed_message_id"])
    )

    print("Saved vs parsed Message-ID mismatches:", mismatch_mask.sum())

    if mismatch_mask.sum() > 0:
        display(
            df.loc[
                mismatch_mask,
                ["doc_id", "label", "message_id", "parsed_message_id"]
            ].head(10)
        )
else:
    df["message_id"] = df["parsed_message_id"]

Saved vs parsed Message-ID mismatches: 0


In [21]:
# Use saved Message-ID as the graph identity when available.
# Fall back to parsed Message-ID only if needed.

df["message_id_for_graph"] = df["message_id"].fillna(df["parsed_message_id"])

df["has_message_id"] = df["message_id_for_graph"].notna()
df["has_references"] = df["references_ids"].apply(lambda x: len(x) > 0)
df["has_in_reply_to"] = df["in_reply_to_ids"].apply(lambda x: len(x) > 0)
df["has_any_thread_reference"] = df["thread_reference_ids"].apply(lambda x: len(x) > 0)

In [22]:
# coverage summary
thread_metadata_summary = pd.DataFrame([
    {
        "field": "Message-ID",
        "documents_with_field": df["has_message_id"].sum(),
        "share": df["has_message_id"].mean(),
    },
    {
        "field": "References",
        "documents_with_field": df["has_references"].sum(),
        "share": df["has_references"].mean(),
    },
    {
        "field": "In-Reply-To",
        "documents_with_field": df["has_in_reply_to"].sum(),
        "share": df["has_in_reply_to"].mean(),
    },
    {
        "field": "References or In-Reply-To",
        "documents_with_field": df["has_any_thread_reference"].sum(),
        "share": df["has_any_thread_reference"].mean(),
    },
])

display(thread_metadata_summary.style.format({"share": "{:.1%}"}))

,field,documents_with_field,share
0,Message-ID,6937,100.0%
1,References,3725,53.7%
2,In-Reply-To,33,0.5%
3,References or In-Reply-To,3738,53.9%
